# Final Reproducibility Analysis
## Machine learning-based leak detection and predefined leak-location classification

**Purpose.** This notebook implements the final analysis protocol reported in the manuscript and Supplementary Table S2.

### Final analysis rules
- Three sensing configurations: pressure-only, vibration-only, and pressure–vibration fusion.
- Final feature representation: 14 pressure features + 14 vibration features = 28 fusion features.
- Detection: 96 observations; fixed 76/20 train/test partition.
- Localization: 48 predefined single-leak observations; fixed 38/10 train/test partition.
- Random state: 42.
- Random Forest: training-only five-fold StratifiedKFold GridSearchCV, accuracy scoring.
- XGBoost, LightGBM and MLP: fixed final configurations; no model-specific GridSearchCV in the final analysis.
- Held-out test observations are not used for model selection.
- Feature importance/redundancy are diagnostic analyses performed after principal model evaluation.
- The notebook does not fabricate experimental run identifiers or wall-clock timestamps.

> **Important:** Place the three frozen Option A CSV files in `data/` (or change `DATA_DIR` below). The notebook reads the processed feature datasets; it does not reconstruct raw sensor signals.

In [ ]:
from pathlib import Path
import sys, platform, warnings, itertools
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef, confusion_matrix
)
from sklearn.inspection import permutation_importance
from scipy.stats import pearsonr, binomtest

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_SPLITS = 5
SCORING = "accuracy"
N_BOOTSTRAP = 10000
CI_LEVEL = 0.95

print("Python:", sys.version)
print("Platform:", platform.platform())

## 1. Input data

Expected frozen Option A files:

1. `OptionA_PressureOnly_Dataset.csv`
2. `OptionA_VibrationOnly_Dataset.csv`
3. `OptionA_CommonTime_Fusion_Dataset.csv`

The expected metadata/label columns include `Scenario`, `detection_label`, and `localization_label`. The notebook also accepts the earlier capitalization convention (`Detection`, `Localization`) by normalizing labels.

In [ ]:
DATA_DIR = Path("data")

FILES = {
    "Pressure": DATA_DIR / "OptionA_PressureOnly_Dataset.csv",
    "Vibration": DATA_DIR / "OptionA_VibrationOnly_Dataset.csv",
    "Fusion": DATA_DIR / "OptionA_CommonTime_Fusion_Dataset.csv",
}

missing = [str(p) for p in FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing input file(s). Put the frozen Option A CSV files in ./data/ or "
        "change DATA_DIR. Missing: " + ", ".join(missing)
    )

datasets = {name: pd.read_csv(path) for name, path in FILES.items()}

for name, df in datasets.items():
    print(f"{name}: shape={df.shape}")
    print(df.columns.tolist())

In [ ]:
PRESSURE_FEATURES = [
    "p_mean","p_std","p_rms","p_var","p_skew","p_kurtosis","p_peak",
    "p_crest_factor","p_energy","p_zero_crossing","p_fft_peak",
    "p_spectral_energy","p_dominant_freq","p_mean_freq"
]

VIBRATION_FEATURES = [
    "v_mean","v_std","v_rms","v_var","v_skew","v_kurtosis","v_peak",
    "v_crest_factor","v_energy","v_zero_crossing","v_fft_peak",
    "v_spectral_energy","v_dominant_freq","v_mean_freq"
]

FEATURE_SETS = {
    "Pressure": PRESSURE_FEATURES,
    "Vibration": VIBRATION_FEATURES,
    "Fusion": PRESSURE_FEATURES + VIBRATION_FEATURES
}

assert len(PRESSURE_FEATURES) == 14
assert len(VIBRATION_FEATURES) == 14
assert len(FEATURE_SETS["Fusion"]) == 28

def normalize_labels(df):
    out = df.copy()
    if "detection_label" not in out.columns and "Detection" in out.columns:
        out["detection_label"] = out["Detection"]
    if "localization_label" not in out.columns and "Localization" in out.columns:
        out["localization_label"] = out["Localization"]
    if "Scenario" not in out.columns and "scenario" in out.columns:
        out["Scenario"] = out["scenario"]
    return out

datasets = {k: normalize_labels(v) for k,v in datasets.items()}

for name, df in datasets.items():
    required = set(FEATURE_SETS[name]) | {"detection_label","localization_label"}
    missing_cols = sorted(required - set(df.columns))
    if missing_cols:
        raise ValueError(f"{name}: missing required columns: {missing_cols}")
    print(name, "feature count:", len(FEATURE_SETS[name]))

## 2. Scenario definition and fixed partitions

Scenario mapping used by the manuscript:

- S1: No leak → detection 0
- S2: Leak 1 → detection 1, localization 1
- S3: Leak 2 → detection 1, localization 2
- S4: Leak 3 → detection 1, localization 3
- S5: Leak 1 + Leak 3 → detection 1, excluded from single-location localization
- S6: All leaks → detection 1, excluded from single-location localization

The archived data do not retain reliable independent experimental-run identifiers. Therefore, this notebook performs the documented observation-level stratified hold-out assessment and does not fabricate run IDs.

In [ ]:
# If labels are already present in the frozen files, retain them.
# Otherwise derive them from Scenario using the documented mapping.
SCENARIO_MAP = {
    "S1": (0, np.nan),
    "S2": (1, 1),
    "S3": (1, 2),
    "S4": (1, 3),
    "S5": (1, np.nan),
    "S6": (1, np.nan),
}

for name, df in datasets.items():
    if "Scenario" in df.columns:
        mapped_det = df["Scenario"].astype(str).map(lambda x: SCENARIO_MAP.get(x, (np.nan,np.nan))[0])
        mapped_loc = df["Scenario"].astype(str).map(lambda x: SCENARIO_MAP.get(x, (np.nan,np.nan))[1])
        if df["detection_label"].isna().any():
            df.loc[df["detection_label"].isna(), "detection_label"] = mapped_det
        if df["localization_label"].isna().any():
            df.loc[df["localization_label"].isna(), "localization_label"] = mapped_loc

for name, df in datasets.items():
    print("\n", name)
    print("Rows:", len(df))
    print("Detection counts:")
    print(df["detection_label"].value_counts(dropna=False).sort_index())
    print("Localization counts:")
    print(df["localization_label"].value_counts(dropna=False).sort_index())

In [ ]:
# Deterministic stratified partitions.
# The manuscript specifies the final partition sizes; this check verifies them.
def make_detection_split(df):
    idx = np.arange(len(df))
    y = df["detection_label"].astype(int).to_numpy()
    tr, te = train_test_split(
        idx, test_size=20, stratify=y, random_state=RANDOM_STATE
    )
    return np.sort(tr), np.sort(te)

def make_localization_split(df):
    loc = df[df["localization_label"].notna()].copy()
    idx = np.arange(len(loc))
    y = loc["localization_label"].astype(int).to_numpy()
    tr, te = train_test_split(
        idx, test_size=10, stratify=y, random_state=RANDOM_STATE
    )
    return loc, np.sort(tr), np.sort(te)

splits = {}
for name, df in datasets.items():
    tr, te = make_detection_split(df)
    if len(df) != 96 or len(tr) != 76 or len(te) != 20:
        raise ValueError(f"{name}: expected 96/76/20, got {len(df)}/{len(tr)}/{len(te)}")
    loc, ltr, lte = make_localization_split(df)
    if len(loc) != 48 or len(ltr) != 38 or len(lte) != 10:
        raise ValueError(f"{name}: expected localization 48/38/10, got {len(loc)}/{len(ltr)}/{len(lte)}")
    splits[name] = {"det_train":tr, "det_test":te, "loc_df":loc, "loc_train":ltr, "loc_test":lte}

print("All three sensing configurations passed the expected 96/76/20 and 48/38/10 checks.")

## 3. Final model configurations

### Random Forest
RF is the only principal model subjected to systematic GridSearchCV in the final analysis. The search uses the 76-observation training partition only, five-fold StratifiedKFold with shuffling, `random_state=42`, and `accuracy` as the selection criterion.

The final configuration is sensing-specific:

- Pressure: 300 trees, `min_samples_split=10`
- Vibration: 100 trees, `min_samples_split=10`
- Fusion: 100 trees, `min_samples_split=2`

### XGBoost
Fixed final configuration: 300 estimators, depth 4, learning rate 0.05, subsample 0.8, column subsampling 0.8.

### LightGBM
Fixed final configuration: 300 estimators, 31 leaves, unlimited depth, learning rate 0.05, subsample 0.8, column subsampling 0.8.

### MLP
Fixed final configuration: hidden layers `(64, 32)`, ReLU, Adam, alpha `1e-4`, learning rate `1e-3`, maximum 1000 iterations, early stopping, validation fraction 0.15, `random_state=42`, with `StandardScaler` in the pipeline.

In [ ]:
RF_FINAL = {
    "Pressure":  {"n_estimators":300, "max_depth":None, "min_samples_split":10, "min_samples_leaf":1, "max_features":"sqrt"},
    "Vibration": {"n_estimators":100, "max_depth":None, "min_samples_split":10, "min_samples_leaf":1, "max_features":"sqrt"},
    "Fusion":    {"n_estimators":100, "max_depth":None, "min_samples_split":2,  "min_samples_leaf":1, "max_features":"sqrt"},
}

RF_GRID = {
    "classifier__n_estimators": [100,200,300],
    "classifier__max_depth": [None,5,10,20],
    "classifier__min_samples_split": [2,5,10],
    "classifier__min_samples_leaf": [1,2,4],
    "classifier__max_features": ["sqrt","log2"],
}

XGB_PARAMS = dict(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, eval_metric="logloss", n_jobs=-1
)

LGB_PARAMS = dict(
    n_estimators=300, num_leaves=31, max_depth=-1, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, verbosity=-1, n_jobs=-1
)

MLP_PARAMS = dict(
    hidden_layer_sizes=(64,32), activation="relu", solver="adam",
    alpha=1e-4, learning_rate_init=1e-3, max_iter=1000,
    early_stopping=True, validation_fraction=0.15,
    random_state=RANDOM_STATE
)

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print("RF search combinations:", np.prod([len(v) for v in RF_GRID.values()]))
print("RF fits per sensing configuration:", np.prod([len(v) for v in RF_GRID.values()]) * N_SPLITS)

In [ ]:
# Optional dependency check for the two gradient-boosting libraries.
try:
    from xgboost import XGBClassifier
    xgb_available = True
except Exception as e:
    xgb_available = False
    print("XGBoost unavailable:", e)

try:
    from lightgbm import LGBMClassifier
    lgb_available = True
except Exception as e:
    lgb_available = False
    print("LightGBM unavailable:", e)

from sklearn.neural_network import MLPClassifier

## 4. Detection: final held-out evaluation

In [ ]:
def detection_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    return {
        "Accuracy": accuracy_score(y_true,y_pred),
        "Precision": precision_score(y_true,y_pred,zero_division=0),
        "Recall": recall_score(y_true,y_pred,zero_division=0),
        "F1": f1_score(y_true,y_pred,zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y_true,y_pred),
        "Specificity": specificity,
        "MCC": matthews_corrcoef(y_true,y_pred),
        "TN":tn,"FP":fp,"FN":fn,"TP":tp
    }

def fit_rf(Xtr,ytr,config):
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
            random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1
        ))
    ])
    search = GridSearchCV(pipe, RF_GRID, scoring=SCORING, cv=cv, n_jobs=-1, refit=True)
    search.fit(Xtr,ytr)
    return search.best_estimator_, search.best_score_, search.best_params_

def fixed_models():
    models = {}
    if xgb_available:
        models["XGBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", XGBClassifier(**XGB_PARAMS))
        ])
    if lgb_available:
        models["LightGBM"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", LGBMClassifier(**LGB_PARAMS))
        ])
    models["MLP"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", MLPClassifier(**MLP_PARAMS))
    ])
    return models

results = []
predictions = {}

for config, df in datasets.items():
    feats = FEATURE_SETS[config]
    tr, te = splits[config]["det_train"], splits[config]["det_test"]
    X = df[feats].to_numpy(float)
    y = df["detection_label"].astype(int).to_numpy()

    # RF: training-only GridSearchCV
    rf, cv_score, best_params = fit_rf(X[tr], y[tr], config)
    pred = rf.predict(X[te])
    key = (config, "Random Forest")
    predictions[key] = {"y_true":y[te], "y_pred":pred, "estimator":rf}
    row = {"Configuration":config,"Model":"Random Forest","CV_Accuracy":cv_score}
    row.update(detection_metrics(y[te],pred))
    results.append(row)

    # Fixed XGB/LGBM/MLP: no GridSearchCV
    for model_name, model in fixed_models().items():
        model.fit(X[tr], y[tr])
        pred = model.predict(X[te])
        key = (config, model_name)
        predictions[key] = {"y_true":y[te], "y_pred":pred, "estimator":model}
        row = {"Configuration":config,"Model":model_name,"CV_Accuracy":np.nan}
        row.update(detection_metrics(y[te],pred))
        results.append(row)

detection_results = pd.DataFrame(results)
display(detection_results[["Configuration","Model","CV_Accuracy","Accuracy","F1","Balanced_Accuracy","Specificity","MCC"]])

## 5. Detection baselines

The manuscript includes:
- majority-class `DummyClassifier`;
- Logistic Regression as a simpler supervised baseline.

These use the same held-out partition. The baselines are not used to alter the principal model configurations.

In [ ]:
baseline_rows = []

for config, df in datasets.items():
    feats = FEATURE_SETS[config]
    tr, te = splits[config]["det_train"], splits[config]["det_test"]
    X = df[feats].to_numpy(float)
    y = df["detection_label"].astype(int).to_numpy()

    dummy = DummyClassifier(strategy="most_frequent")
    dummy.fit(X[tr], y[tr])
    pred = dummy.predict(X[te])
    r = {"Configuration":config,"Baseline":"Majority-class Dummy"}
    r.update(detection_metrics(y[te],pred))
    baseline_rows.append(r)

    lr = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
    ])
    lr.fit(X[tr], y[tr])
    pred = lr.predict(X[te])
    r = {"Configuration":config,"Baseline":"Logistic Regression"}
    r.update(detection_metrics(y[te],pred))
    baseline_rows.append(r)

baseline_results = pd.DataFrame(baseline_rows)
display(baseline_results[["Configuration","Baseline","Accuracy","Balanced_Accuracy","Precision","Recall","Specificity","F1","MCC"]])

## 6. Localization: predefined single-leak-location classification

In [ ]:
def localization_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true,y_pred),
        "Macro_Precision": precision_score(y_true,y_pred,average="macro",zero_division=0),
        "Macro_Recall": recall_score(y_true,y_pred,average="macro",zero_division=0),
        "Macro_F1": f1_score(y_true,y_pred,average="macro",zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y_true,y_pred),
        "MCC": matthews_corrcoef(y_true,y_pred)
    }

loc_results = []
loc_predictions = {}

for config, df in datasets.items():
    feats = FEATURE_SETS[config]
    loc_df = splits[config]["loc_df"]
    tr, te = splits[config]["loc_train"], splits[config]["loc_test"]
    X = loc_df[feats].to_numpy(float)
    y = loc_df["localization_label"].astype(int).to_numpy()

    # RF uses the configuration-specific final parameters selected in the final detection workflow.
    rf_params = RF_FINAL[config]
    rf = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
            **rf_params, class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
    rf.fit(X[tr], y[tr])
    pred = rf.predict(X[te])
    loc_predictions[(config,"Random Forest")] = {"y_true":y[te],"y_pred":pred}
    r={"Configuration":config,"Model":"Random Forest"}
    r.update(localization_metrics(y[te],pred)); loc_results.append(r)

    for model_name, model in fixed_models().items():
        model.fit(X[tr], y[tr])
        pred = model.predict(X[te])
        loc_predictions[(config,model_name)]={"y_true":y[te],"y_pred":pred}
        r={"Configuration":config,"Model":model_name}
        r.update(localization_metrics(y[te],pred)); loc_results.append(r)

localization_results = pd.DataFrame(loc_results)
display(localization_results[["Configuration","Model","Accuracy","Macro_Precision","Macro_Recall","Macro_F1"]])

## 7. Confusion matrices

All confusion matrices below are generated directly from the final held-out predictions used for the reported performance metrics. They are not separate evaluation experiments.

In [ ]:
print("DETECTION CONFUSION MATRICES")
for key, obj in predictions.items():
    print("\n", key)
    print(confusion_matrix(obj["y_true"], obj["y_pred"], labels=[0,1]))

print("\nLOCALIZATION CONFUSION MATRICES")
for key, obj in loc_predictions.items():
    print("\n", key)
    print(confusion_matrix(obj["y_true"], obj["y_pred"], labels=[1,2,3]))

## 8. Bootstrap uncertainty for the frozen held-out detection predictions

In [ ]:
def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=N_BOOTSTRAP, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    vals = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0,n,n)
        vals[i] = metric_fn(y_true[idx], y_pred[idx])
    alpha = 1 - CI_LEVEL
    return np.nanpercentile(vals, [100*alpha/2, 100*(1-alpha/2)])

metric_functions = {
    "Accuracy": accuracy_score,
    "Precision": lambda a,b: precision_score(a,b,zero_division=0),
    "Recall": lambda a,b: recall_score(a,b,zero_division=0),
    "F1": lambda a,b: f1_score(a,b,zero_division=0),
    "Balanced_Accuracy": balanced_accuracy_score,
    "MCC": matthews_corrcoef,
}

bootstrap_rows=[]
for key,obj in predictions.items():
    config, model = key
    for metric, fn in metric_functions.items():
        lo, hi = bootstrap_ci(obj["y_true"], obj["y_pred"], fn)
        bootstrap_rows.append({
            "Configuration":config,"Model":model,"Metric":metric,
            "Lower_95":lo,"Upper_95":hi
        })
bootstrap_results=pd.DataFrame(bootstrap_rows)
display(bootstrap_results)

## 9. Statistical comparison

The final manuscript specifies:
- binary detection: exact McNemar testing with Holm correction;
- multiclass localization: paired permutation testing of accuracy differences;
- α = 0.05.

These tests compare the same held-out observations and do not refit models.

In [ ]:
def exact_mcnemar(y_true, pred_a, pred_b):
    a_correct = pred_a == y_true
    b_correct = pred_b == y_true
    b01 = np.sum(a_correct & ~b_correct)
    b10 = np.sum(~a_correct & b_correct)
    n = b01 + b10
    if n == 0:
        return 1.0, b01, b10
    p = binomtest(min(b01,b10), n=n, p=0.5, alternative="two-sided").pvalue
    return p, b01, b10

det_keys=list(predictions.keys())
pairs=[]
for a,b in itertools.combinations(det_keys,2):
    p,b01,b10=exact_mcnemar(
        predictions[a]["y_true"],
        predictions[a]["y_pred"],
        predictions[b]["y_pred"]
    )
    pairs.append({"A":a,"B":b,"raw_p":p,"discordant_A_not_B":b01,"discordant_B_not_A":b10})

det_stats=pd.DataFrame(pairs)
if len(det_stats):
    order=np.argsort(det_stats["raw_p"].to_numpy())
    holm=np.empty(len(det_stats))
    m=len(det_stats)
    running=0
    for rank,idx in enumerate(order):
        adj=min(1,(m-rank)*det_stats.loc[idx,"raw_p"])
        running=max(running,adj)
        holm[idx]=running
    det_stats["Holm_p"]=holm
    det_stats["Significant_alpha_0.05"]=det_stats["Holm_p"]<0.05

display(det_stats)

In [ ]:
def paired_permutation_accuracy(y_true, pred_a, pred_b, n_perm=10000, seed=RANDOM_STATE):
    rng=np.random.default_rng(seed)
    d=(pred_a==y_true).astype(int)-(pred_b==y_true).astype(int)
    observed=abs(d.mean())
    count=0
    for _ in range(n_perm):
        signs=rng.choice([-1,1], size=len(d))
        if abs((d*signs).mean()) >= observed:
            count += 1
    return (count+1)/(n_perm+1)

loc_keys=list(loc_predictions.keys())
loc_pairs=[]
for a,b in itertools.combinations(loc_keys,2):
    ya=loc_predictions[a]["y_true"]
    yb=loc_predictions[b]["y_true"]
    if not np.array_equal(ya,yb):
        raise ValueError("Localization test labels are not aligned.")
    p=paired_permutation_accuracy(ya,loc_predictions[a]["y_pred"],loc_predictions[b]["y_pred"])
    loc_pairs.append({"A":a,"B":b,"raw_p":p})

loc_stats=pd.DataFrame(loc_pairs)
if len(loc_stats):
    order=np.argsort(loc_stats["raw_p"].to_numpy())
    holm=np.empty(len(loc_stats)); m=len(loc_stats); running=0
    for rank,idx in enumerate(order):
        adj=min(1,(m-rank)*loc_stats.loc[idx,"raw_p"])
        running=max(running,adj); holm[idx]=running
    loc_stats["Holm_p"]=holm
    loc_stats["Significant_alpha_0.05"]=loc_stats["Holm_p"]<0.05

display(loc_stats)

## 10. Feature-importance and redundancy diagnostics

Permutation importance is calculated as a model- and dataset-dependent predictive contribution measure. It is not interpreted as causal importance.

The redundancy analysis uses the predefined criterion `|Pearson r| >= 0.90`. No feature is removed and no reported model is retrained as a consequence of this diagnostic.

In [ ]:
# Use the final Random Forest estimators from the detection evaluation.
importance_rows=[]
for key,obj in predictions.items():
    config,model=key
    if model != "Random Forest":
        continue
    df=datasets[config]
    tr,te=splits[config]["det_train"],splits[config]["det_test"]
    X_test=df.iloc[te][FEATURE_SETS[config]]
    y_test=df.iloc[te]["detection_label"].astype(int)
    pi=permutation_importance(
        obj["estimator"], X_test, y_test,
        n_repeats=10, random_state=RANDOM_STATE, scoring="accuracy"
    )
    for f,mean,std in sorted(
        zip(FEATURE_SETS[config],pi.importances_mean,pi.importances_std),
        key=lambda z:z[1], reverse=True
    ):
        importance_rows.append({
            "Configuration":config,"Model":model,
            "Feature":f,"Permutation_Importance":mean,
            "Permutation_SD":std
        })

importance_results=pd.DataFrame(importance_rows)
display(importance_results.head(20))

In [ ]:
redundancy_rows=[]
for config,df in datasets.items():
    X=df[FEATURE_SETS[config]]
    corr=X.corr(method="pearson")
    for i,j in itertools.combinations(FEATURE_SETS[config],2):
        r=corr.loc[i,j]
        if pd.notna(r) and abs(r)>=0.90:
            redundancy_rows.append({
                "Configuration":config,
                "Feature_A":i,"Feature_B":j,"Pearson_r":r
            })

redundancy_results=pd.DataFrame(redundancy_rows)
print("Highly correlated pairs:", len(redundancy_results))
display(redundancy_results)

print("Expected manuscript summary: 46 pairs overall (8 pressure, 15 vibration, 23 fusion).")

## 11. Final manuscript consistency checks

This section checks the structural requirements of the final manuscript. Numerical values are calculated from the supplied frozen datasets and final model protocol; they are not manually inserted into the analysis.

In [ ]:
print("FINAL REPRODUCIBILITY CHECK")
print("="*70)

checks = {
    "Three sensing configurations": len(datasets)==3,
    "14 pressure features": len(PRESSURE_FEATURES)==14,
    "14 vibration features": len(VIBRATION_FEATURES)==14,
    "28 fusion features": len(FEATURE_SETS["Fusion"])==28,
    "Detection 96/76/20 for all configurations": all(
        len(datasets[c])==96 and len(splits[c]["det_train"])==76 and len(splits[c]["det_test"])==20
        for c in datasets
    ),
    "Localization 48/38/10 for all configurations": all(
        len(splits[c]["loc_df"])==48 and len(splits[c]["loc_train"])==38 and len(splits[c]["loc_test"])==10
        for c in datasets
    ),
    "Random state = 42": RANDOM_STATE==42,
    "Five-fold CV": N_SPLITS==5,
    "RF selection scoring = accuracy": SCORING=="accuracy",
    "RF grid = 216 combinations": int(np.prod([len(v) for v in RF_GRID.values()]))==216,
    "Bootstrap replicates = 10000": N_BOOTSTRAP==10000,
    "Redundancy threshold = 0.90": True,
}

for k,v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}: {k}")

if not all(checks.values()):
    raise AssertionError("One or more final reproducibility checks failed.")

## 12. Interpretation safeguards

This notebook is intentionally limited to the final manuscript protocol.

- The 96 detection observations are analysis windows, not 96 independent physical experiments.
- No unavailable experimental run IDs or wall-clock timestamps are fabricated.
- Multi-leak scenarios are retained for detection and excluded from single-location localization.
- Localization is classification among three predefined experimental leak locations, not continuous coordinate estimation.
- Feature importance is model/data dependent and not causal.
- Redundant features are identified diagnostically; no retrospective feature removal or retraining is performed.
- The supplementary Gaussian perturbation analysis is a feature-space robustness diagnostic and is not implemented here as raw sensor-noise validation.
- The study remains an offline controlled-laboratory benchmark; this notebook does not establish field deployment, real-time performance, or scalability.